# Readout quickstart

This notebook builds two small, independent readout examples from public TensorDSLab pieces. The numbers are illustrative. Application packages usually provide their own axes, kernels, and workflow wrappers.

## 1. Imports

We begin with Torch, Matplotlib, and the public TensorCore and TensorDSLab objects used below.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

from tensor_core import (
    CountCoordinates,
    LabelCoordinates,
    NonnegativeInteger,
    OffsetAxis,
    OffsetCoordinates,
    RegularCoordinates,
    Threefry4x32,
)

from tensor_dslab import (
    AnalogGain,
    AnalogGainSpec,
    AnalogWaveform,
    AnalogWaveformConfig,
    AnalogWaveformKernels,
    AnalogWaveformSpec,
    BitDepth,
    BitDepthSpec,
    ChannelAxis,
    Charge,
    ChargeConfig,
    ChargeKernels,
    ChargeSpec,
    DigitizedWaveform,
    DigitizedWaveformConfig,
    DigitizedWaveformKernels,
    DigitizedWaveformSpec,
    EncodedWaveform,
    EncodedWaveformConfig,
    EncodedWaveformKernels,
    EncodedWaveformSpec,
    ExampleAxis,
    FrequencyAxis,
    InputMaximum,
    InputMaximumSpec,
    InputMinimum,
    InputMinimumSpec,
    NoiseWaveform,
    NoiseWaveformConfig,
    NoiseWaveformKernels,
    NoiseWaveformSpec,
    Photoelectrons,
    PhotoelectronsSpec,
    PostTriggerSamples,
    PostTriggerSamplesSpec,
    PowerSpectralDensity,
    PowerSpectralDensitySpec,
    PreTriggerSamples,
    PreTriggerSamplesSpec,
    PulseResponse,
    PulseResponseSpec,
    PureWaveform,
    PureWaveformConfig,
    PureWaveformKernels,
    PureWaveformSpec,
    ReleaseThresholdCode,
    ReleaseThresholdCodeSpec,
    RequiredTimeOverSamples,
    RequiredTimeOverSamplesSpec,
    TimeAxis,
    TriggerThresholdCode,
    TriggerThresholdCodeSpec,
    unit_registry,
)

## 2. Axes

An Example axis batches two independent waveform realizations, the Channel axis names three sensors, and the Time axis gives the last tensor dimension a physical spacing. We deliberately use both examples and all three channels so it is clear that one Product call processes the complete tensor together. Every Product below uses this same ordered domain. The Frequency axis describes the PSD bins used during preparation; it is not a Product dimension.

In [ ]:
device = torch.device("cpu")
field_dtype = torch.float32

example_axis = ExampleAxis(
    coordinates=CountCoordinates(count=2),
)

channel_axis = ChannelAxis(
    coordinates=LabelCoordinates(
        labels=("sensor-0", "sensor-1", "sensor-2"),
    ),
)

time_axis = TimeAxis(
    coordinates=RegularCoordinates(
        start=0,
        step=1,
        count=5000,
    ),
    coordinate_scale=2.0,
    unit=unit_registry.Unit("ns"),
)

frequency_axis = FrequencyAxis(
    coordinates=RegularCoordinates(
        start=0,
        step=1,
        count=2501,
    ),
    coordinate_scale=0.1,
    unit=unit_registry.Unit("MHz"),
)

axes = (
    example_axis,
    channel_axis,
    time_axis,
)

## 3. Plotting setup

A small presentation-only helper keeps the seven Product grids consistent. Each figure uses columns for the two independent examples and rows for the three sensors, so every subplot corresponds to one exact `(example, channel)` lane. A Product keeps one color across all six lanes. Titles and row labels make legends unnecessary, and the helper never constructs or changes a Product.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

channel_labels = channel_axis.coordinates.labels
time_ns = [
    float(time_axis.quantity_at(index).to("ns").magnitude)
    for index in range(time_axis.size)
]
product_figures = []


def plot_product(
    product,
    *,
    title,
    color,
    step,
    suppression_code=None,
):
    figure, plot_axes = plt.subplots(
        3,
        2,
        figsize=(13, 8.5),
        sharex=True,
        squeeze=False,
    )
    for channel_index, channel_label in enumerate(channel_labels):
        for example_index in range(example_axis.size):
            plot_axis = plot_axes[channel_index, example_index]
            values = (
                product.tensor[example_index, channel_index]
                .detach()
                .cpu()
                .tolist()
            )
            if suppression_code is not None:
                values = [
                    float("nan") if value == suppression_code else value
                    for value in values
                ]
            if step:
                plot_axis.step(
                    time_ns,
                    values,
                    where="post",
                    color=color,
                    alpha=0.72,
                    linewidth=0.9,
                )
            else:
                plot_axis.plot(
                    time_ns,
                    values,
                    color=color,
                    alpha=0.72,
                    linewidth=0.9,
                )
            if channel_index == 0:
                plot_axis.set_title(f"Example {example_index}")
            if example_index == 0:
                plot_axis.set_ylabel(channel_label)
            if channel_index == channel_axis.size - 1:
                plot_axis.set_xlabel("Time (ns)")
    figure.suptitle(title)
    figure.subplots_adjust(
        left=0.08,
        right=0.98,
        bottom=0.08,
        top=0.91,
        hspace=0.18,
        wspace=0.12,
    )
    product_figures.append(figure)
    plt.show()

## 4. Photoelectron values

First we prepare the source counts as an ordinary integer tensor. Example 0 keeps the four familiar deposits, while Example 1 uses four different deposits across the same three sensors. The two examples are independent waveform realizations, not adjacent windows or sequential state. Every pulse remains inside the 10,000 ns window. The copied before-image lets the later execution check show that Product creation leaves this input unchanged.

In [ ]:
photoelectron_values = torch.zeros(
    tuple(axis.size for axis in axes),
    dtype=torch.int64,
    device=device,
)
photoelectron_values[0, 0, 100] = 1
photoelectron_values[0, 0, 3700] = 4
photoelectron_values[0, 1, 1300] = 2
photoelectron_values[0, 2, 2500] = 3
photoelectron_values[1, 0, 800] = 2
photoelectron_values[1, 1, 2000] = 4
photoelectron_values[1, 1, 3500] = 1
photoelectron_values[1, 2, 2900] = 3
photoelectron_values_before = photoelectron_values.clone()

## 5. Photoelectrons

Now the semantic layer names the tensor's axes, device, dtype, and avalanche unit. One Product call processes both independent examples and all three sensors together. `Photoelectrons` is the already-produced source Product for this demonstration; a real application may obtain the same public Product from simulation, measurement, or another transformation.

In [ ]:
photoelectrons_spec = PhotoelectronsSpec(
    axes=axes,
    device=device,
    dtype=torch.int64,
    unit=unit_registry.Unit("avalanche"),
)

photoelectrons = Photoelectrons(
    tensor=photoelectron_values,
    spec=photoelectrons_spec,
)

The first local grid shows each example and sensor lane separately, including all eight sparse source deposits. Step drawing is appropriate because each integer count belongs to one discrete time sample.

In [ ]:
plot_product(
    photoelectrons,
    title="Photoelectrons",
    color="tab:blue",
    step=True,
)

## 6. Charge

An empty ChargeKernels collection leaves these source counts unchanged apart from the requested floating representation. This is a deliberate minimal starting point. Users may later add physical Charge mechanisms with public Kernels.

In [ ]:
charge_spec = ChargeSpec(
    axes=axes,
    device=device,
    dtype=field_dtype,
    unit=unit_registry.Unit("avalanche"),
)

charge_config = ChargeConfig(
    spec=charge_spec,
    kernels=ChargeKernels(members=()),
    correlated_avalanche_generations=NonnegativeInteger(value=0),
)

rng = Threefry4x32(seed=2026)

charge = Charge.create(
    sources=(photoelectrons,),
    config=charge_config,
    rng=rng,
)

With no Charge mechanisms enabled, this step view matches the source deposits while expressing them as floating avalanche charge.

In [ ]:
plot_product(
    charge,
    title="Charge (avalanche)",
    color="tab:orange",
    step=True,
)

## 7. Pulse mathematics

The illustrative pulse is prepared transparently from a Gaussian and two error functions. These familiar numerical values are illustrative rather than calibrated, and the 1,011 coefficients fit fully inside the chosen window.

In [ ]:
pulse_support_ns = 2020.27
pulse_coefficient_count = math.ceil(
    pulse_support_ns / time_axis.coordinate_scale
)
pulse_offsets = torch.arange(
    pulse_coefficient_count,
    dtype=field_dtype,
    device=device,
)
pulse_time_ns = pulse_offsets * time_axis.coordinate_scale
pulse_x = pulse_time_ns - 232.89

pulse_gaussian = torch.exp(
    -(pulse_x**2) / (2.0 * 507.72**2)
) / math.sqrt(2.0 * math.pi * 507.72**2)
pulse_first = 1.0 + torch.erf(
    (pulse_x - (-81.92)) / (math.sqrt(2.0) * 147.28)
)
pulse_second = 1.0 + torch.erf(
    (pulse_x - (-176.50)) / (math.sqrt(2.0) * 45.69)
)

pulse_raw = pulse_gaussian * pulse_first * pulse_second
pulse_values = (
    pulse_raw / torch.max(torch.abs(pulse_raw)) * -14.5912372
)

## 8. Pure waveform

The semantic construction gives those prepared coefficients a Time-relative operation axis and a voltage-per-avalanche meaning. `PulseResponse` then becomes the one Kernel used by `PureWaveform.create`; its empty conditioning geometry shares the response across all three sensors.

In [ ]:
pulse_time_axis = OffsetAxis(
    coordinates=OffsetCoordinates(offsets=tuple(range(1011))),
    relative_to=TimeAxis,
)

pulse_response = PulseResponse(
    tensor=pulse_values,
    spec=PulseResponseSpec(
        conditioning_axes=(),
        operation_axes=(pulse_time_axis,),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV / avalanche"),
    ),
)

pure_waveform_config = PureWaveformConfig(
    spec=PureWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=PureWaveformKernels(
        members=(pulse_response,),
    ),
)

pure_waveform = PureWaveform.create(
    sources=(charge,),
    config=pure_waveform_config,
)

The pure waveform view turns each deposit into the illustrative negative-going pulse response. Thin translucent lines keep overlapping sensor pulses distinguishable.

In [ ]:
plot_product(
    pure_waveform,
    title="Pure (mV)",
    color="tab:green",
    step=False,
)

## 9. PSD values

Three literal rows prepare a different illustrative electronic-noise spectrum for each sensor. This cell handles only the ordinary tensor values; the next cell gives them their Channel and Frequency semantics.

In [ ]:
psd_sensor_0 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((625,), 0.012, dtype=field_dtype, device=device),
        torch.full((1875,), 0.004, dtype=field_dtype, device=device),
    )
)
psd_sensor_1 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((938,), 0.008, dtype=field_dtype, device=device),
        torch.full((1562,), 0.016, dtype=field_dtype, device=device),
    )
)
psd_sensor_2 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((1250,), 0.020, dtype=field_dtype, device=device),
        torch.full((1250,), 0.006, dtype=field_dtype, device=device),
    )
)

psd_values = torch.stack(
    (
        psd_sensor_0,
        psd_sensor_1,
        psd_sensor_2,
    )
)

## 10. Noise waveform

`PowerSpectralDensitySpec` now says that the prepared rows are conditioned by Channel and operate over Frequency. `NoiseWaveform.create` has no source Product here; preparation checks that the frequency grid matches the output Time axis before the seeded draw.

In [ ]:
psd_spec = PowerSpectralDensitySpec(
    conditioning_axes=(channel_axis,),
    operation_axes=(frequency_axis,),
    device=device,
    dtype=field_dtype,
    unit=unit_registry.Unit("mV ** 2"),
)

power_spectral_density = PowerSpectralDensity(
    tensor=psd_values,
    spec=psd_spec,
)

noise_waveform_config = NoiseWaveformConfig(
    spec=NoiseWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=NoiseWaveformKernels(
        members=(power_spectral_density,),
    ),
)

noise_waveform = NoiseWaveform.create(
    sources=(),
    config=noise_waveform_config,
    rng=rng,
)

This local view shows the seeded PSD-driven noise by itself, making the different sensor spectra visible before they are combined with the pulse response.

In [ ]:
plot_product(
    noise_waveform,
    title="Noise (mV)",
    color="tab:red",
    step=False,
)

## 11. Analog waveform

AnalogWaveform combines the pure response and the electronic noise. Empty saturation Kernels keep this example focused on ordinary source composition.

In [ ]:
analog_waveform_config = AnalogWaveformConfig(
    spec=AnalogWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=AnalogWaveformKernels(members=()),
)

analog_waveform = AnalogWaveform.create(
    sources=(
        pure_waveform,
        noise_waveform,
    ),
    config=analog_waveform_config,
)

The analog view now shows the pulse response and electronic noise together, immediately after their public Product composition.

In [ ]:
plot_product(
    analog_waveform,
    title="Analog (mV)",
    color="tab:purple",
    step=False,
)

## 12. Digitizer values

Four scalar tensors hold the illustrative bit depth, input interval, and linear gain. The -80 mV to 20 mV interval keeps the familiar pulse visible without rail clipping; these values are a demo choice, not calibration.

In [ ]:
bit_depth_value = torch.tensor(12, dtype=torch.int16, device=device)
input_minimum_value = torch.tensor(
    -80.0, dtype=field_dtype, device=device
)
input_maximum_value = torch.tensor(
    20.0, dtype=field_dtype, device=device
)
analog_gain_value = torch.tensor(
    1.0, dtype=field_dtype, device=device
)

## 13. Digitized waveform

The semantic cell wraps each prepared scalar in its public Kernel Spec and Kernel, then collects them in `DigitizedWaveformConfig`. These Kernels apply globally here, while an application may condition the same public coefficient types on Channel or Example axes when its hardware varies.

In [ ]:
bit_depth = BitDepth(
    tensor=bit_depth_value,
    spec=BitDepthSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int16,
    ),
)

input_minimum = InputMinimum(
    tensor=input_minimum_value,
    spec=InputMinimumSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
)

input_maximum = InputMaximum(
    tensor=input_maximum_value,
    spec=InputMaximumSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
)

analog_gain = AnalogGain(
    tensor=analog_gain_value,
    spec=AnalogGainSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit(""),
    ),
)

digitized_waveform_config = DigitizedWaveformConfig(
    spec=DigitizedWaveformSpec(
        axes=axes,
        device=device,
        dtype=torch.int32,
        unit=unit_registry.Unit(""),
    ),
    kernels=DigitizedWaveformKernels(
        members=(
            bit_depth,
            input_minimum,
            input_maximum,
            analog_gain,
        ),
    ),
)

digitized_waveform = DigitizedWaveform.create(
    sources=(analog_waveform,),
    config=digitized_waveform_config,
)

The digitized step view shows the same analog signals as integer ADC codes. The selected illustrative interval keeps the waveforms away from both rails.

In [ ]:
plot_product(
    digitized_waveform,
    title="ADC code",
    color="tab:brown",
    step=True,
)

## 14. Encoding values

Five scalar tensors prepare the illustrative raw-ZLE trigger, release, time-over, and padding policy. Keeping these ordinary values separate makes the semantic Kernel construction in the next cell easier to scan.

In [ ]:
trigger_threshold_value = torch.tensor(
    2500, dtype=torch.int64, device=device
)
release_threshold_value = torch.tensor(
    2800, dtype=torch.int64, device=device
)
required_time_over_value = torch.tensor(
    3, dtype=torch.int64, device=device
)
pre_trigger_value = torch.tensor(
    25, dtype=torch.int64, device=device
)
post_trigger_value = torch.tensor(
    50, dtype=torch.int64, device=device
)

## 15. Encoded waveform

Each policy value now receives its exact public Spec and Kernel before `EncodedWaveform.create` selects retained ADC samples. Retained values remain exact codes; the configured negative value marks samples that were not retained. The global settings are illustrative rather than detector calibration.

In [ ]:
trigger_threshold_code = TriggerThresholdCode(
    tensor=trigger_threshold_value,
    spec=TriggerThresholdCodeSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
release_threshold_code = ReleaseThresholdCode(
    tensor=release_threshold_value,
    spec=ReleaseThresholdCodeSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
required_time_over_samples = RequiredTimeOverSamples(
    tensor=required_time_over_value,
    spec=RequiredTimeOverSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
pre_trigger_samples = PreTriggerSamples(
    tensor=pre_trigger_value,
    spec=PreTriggerSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
post_trigger_samples = PostTriggerSamples(
    tensor=post_trigger_value,
    spec=PostTriggerSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)

encoded_waveform_config = EncodedWaveformConfig(
    spec=EncodedWaveformSpec(
        axes=axes,
        device=device,
        dtype=torch.int32,
        unit=unit_registry.Unit(""),
        suppression_code=-1,
    ),
    kernels=EncodedWaveformKernels(
        members=(
            trigger_threshold_code,
            release_threshold_code,
            required_time_over_samples,
            pre_trigger_samples,
            post_trigger_samples,
        ),
    ),
)

encoded_waveform = EncodedWaveform.create(
    sources=(digitized_waveform,),
    config=encoded_waveform_config,
)

The final local view leaves suppressed regions blank while showing every retained ADC interval unchanged. Only this presentation call supplies the negative suppression code; the Product tensor itself is not modified.

In [ ]:
plot_product(
    encoded_waveform,
    title="Retained ADC code",
    color="tab:pink",
    step=True,
    suppression_code=encoded_waveform.spec.suppression_code,
)

## 16. Shared shape

Each Product represents a different quantity or transformation, but all seven use the same `(example, channel, time)` domain. These light checks make that shared shape visible without repeating Product validation.

In [ ]:
expected_shape = tuple(axis.size for axis in axes)

assert photoelectrons.tensor.shape == expected_shape
assert charge.tensor.shape == expected_shape
assert pure_waveform.tensor.shape == expected_shape
assert noise_waveform.tensor.shape == expected_shape
assert analog_waveform.tensor.shape == expected_shape
assert digitized_waveform.tensor.shape == expected_shape
assert encoded_waveform.tensor.shape == expected_shape